<a href="https://colab.research.google.com/github/MeenakshiRajpurohit/ISE-201-Math-Foundations-for-Decision-Data-Sciences/blob/main/student_notes_VQA_MODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 — install first
!pip install -q transformers datasets pillow pandas tabulate accelerate

In [ ]:
# ============================================================
#  VQA INFERENCE — Student Notes / Finance Dataset
#  Works on: Google Colab, Kaggle Notebooks
#  Model: Qwen2-VL-2B-Instruct
#  No local files needed — streams from HuggingFace
# ============================================================

# ── CELL 1: Install dependencies ────────────────────────────
# !pip install -q transformers datasets pillow pandas tabulate torch torchvision accelerate


# ── CELL 2: Imports ─────────────────────────────────────────
import os
import re
import torch
import pandas as pd
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from datasets import load_dataset

# ── CELL 3: Config ───────────────────────────────────────────
MODEL_NAME      = "Qwen/Qwen2-VL-2B-Instruct"
FINETUNED_CKPT  = None        # set to your LoRA path if using fine-tuned model
MAX_SAMPLES     = 50          # how many samples to run inference on
MAX_NEW_TOKENS  = 64
SCORE_THRESHOLD = 0.5         # F1 cutoff for "well predicted"
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


# ── CELL 4: Drop-in replacement for qwen_vl_utils ────────────
def process_vision_info(messages):
    """No external dependency version of qwen_vl_utils.process_vision_info"""
    image_inputs = []
    for msg in messages:
        content = msg.get("content", [])
        if isinstance(content, list):
            for item in content:
                if item.get("type") == "image":
                    img = item.get("image")
                    if isinstance(img, str):
                        img = Image.open(img).convert("RGB")
                    elif not isinstance(img, Image.Image):
                        img = Image.fromarray(img).convert("RGB")
                    image_inputs.append(img)
    return image_inputs, []   # images, videos


# ── CELL 5: Load Dataset from HuggingFace ────────────────────
def load_dataset_hf():
    """
    Tries multiple dataset sources in order:
      1. nbroad/DocVQA  (student notes)
      2. sujet-ai/Sujet-Finance-QA-Vision-100k  (finance VQA — your project dataset)
    Returns list of {"image": PIL, "question": str, "answer": str}
    """
    candidates = [
        {
            "repo":    "HuggingFaceM4/DocumentVQA",
            "split":   "validation",
            "img_key": "image",
            "q_key":   "question",
            "a_key":   "answers",   # list — will take first element
        },
        {
            "repo":    "sujet-ai/Sujet-Finance-QA-Vision-100k",
            "split":   "train",
            "img_key": "image",
            "q_key":   "question",
            "a_key":   "answer",
        },
        {
            "repo":    "naver-clova-ix/docvqa_train_val_test",
            "split":   "validation",
            "img_key": "image",
            "q_key":   "question",
            "a_key":   "answers",
        },
    ]

    for cfg in candidates:
        try:
            print(f"  Trying: {cfg['repo']} ...")
            ds = load_dataset(cfg["repo"], split=cfg["split"], streaming=False, trust_remote_code=True)
            print(f"  ✅ Loaded {len(ds)} samples from {cfg['repo']}")
            print(f"  Keys: {list(ds[0].keys())}")

            samples = []
            for item in ds.select(range(min(MAX_SAMPLES, len(ds)))):
                img = item[cfg["img_key"]]
                if not isinstance(img, Image.Image):
                    img = Image.fromarray(img).convert("RGB")
                else:
                    img = img.convert("RGB")

                ans = item.get(cfg["a_key"], "")
                if isinstance(ans, list):
                    ans = ans[0] if ans else ""

                samples.append({
                    "image":    img,
                    "question": str(item[cfg["q_key"]]),
                    "answer":   str(ans),
                })
            return samples

        except Exception as e:
            print(f"  ❌ Failed: {e}")

    raise RuntimeError("All dataset sources failed. Check your internet connection.")


print("📂 Loading dataset...")
samples = load_dataset_hf()
print(f"\nSample 0 question : {samples[0]['question']}")
print(f"Sample 0 answer   : {samples[0]['answer']}")
samples[0]["image"]   # shows image in notebook


# ── CELL 6: Load Model ────────────────────────────────────────
def load_model():
    print(f"\n🔄 Loading {MODEL_NAME} ...")
    processor = AutoProcessor.from_pretrained(
        MODEL_NAME, trust_remote_code=True
    )
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        device_map="auto" if DEVICE == "cuda" else None,
        trust_remote_code=True,
    )

    if FINETUNED_CKPT and os.path.exists(FINETUNED_CKPT):
        from peft import PeftModel
        print(f"  🔧 Applying LoRA from: {FINETUNED_CKPT}")
        model = PeftModel.from_pretrained(model, FINETUNED_CKPT)
        model = model.merge_and_unload()
        print("  ✅ LoRA merged")

    model.eval()
    if DEVICE == "cpu":
        model = model.to(DEVICE)
    print(f"  ✅ Model ready on {DEVICE}")
    return model, processor


model, processor = load_model()


# ── CELL 7: Inference ────────────────────────────────────────

def predict(model, processor, image: Image.Image, question: str) -> str:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text":  question},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    # Pass image directly — skip process_vision_info entirely
    inputs = processor(
        text=[text],
        images=[image],        # ← direct PIL image, no wrapper function
        padding=True,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)

    trimmed = [
        out[len(inp):]
        for inp, out in zip(inputs.input_ids, output_ids)
    ]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()


# ── CELL 8: Scoring ──────────────────────────────────────────
def normalize(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    articles = {"a", "an", "the"}
    return " ".join(t for t in text.split() if t not in articles)

def exact_match(pred, gold):
    return float(normalize(pred) == normalize(gold))

def token_f1(pred, gold):
    p_toks = normalize(pred).split()
    g_toks = normalize(gold).split()
    if not p_toks or not g_toks:
        return 0.0
    common = set(p_toks) & set(g_toks)
    if not common:
        return 0.0
    prec = len(common) / len(p_toks)
    rec  = len(common) / len(g_toks)
    return 2 * prec * rec / (prec + rec)


# ── CELL 9: Run Inference Loop ────────────────────────────────
print(f"\n🚀 Running inference on {len(samples)} samples...\n")

results = []
for i, s in enumerate(samples):
    pred = predict(model, processor, s["image"], s["question"])
    gold = s["answer"]
    em   = exact_match(pred, gold)
    f1   = token_f1(pred, gold)

    results.append({
        "idx":          i,
        "question":     s["question"],
        "ground_truth": gold,
        "prediction":   pred,
        "exact_match":  em,
        "f1_score":     round(f1, 3),
        "correct":      f1 >= SCORE_THRESHOLD,
    })

    status = "✅" if f1 >= SCORE_THRESHOLD else "❌"
    print(f"[{i+1:>3}/{len(samples)}] {status}  F1={f1:.2f}  Q: {s['question'][:55]}")
    print(f"         Gold: {gold[:50]}")
    print(f"         Pred: {pred[:50]}\n")


# ── CELL 10: Results Summary ─────────────────────────────────
df = pd.DataFrame(results)

print("\n" + "="*65)
print("📊  OVERALL METRICS")
print("="*65)
print(f"  Total samples       : {len(df)}")
print(f"  Exact Match (EM)    : {df['exact_match'].mean()*100:.2f}%")
print(f"  Token F1            : {df['f1_score'].mean()*100:.2f}%")
print(f"  Correct (F1≥{SCORE_THRESHOLD})    : {df['correct'].sum()} / {len(df)}  ({df['correct'].mean()*100:.1f}%)")


# ── CELL 11: Well-Predicted Results Table ────────────────────
well = df[df["correct"]].reset_index(drop=True)
print(f"\n{'='*65}")
print(f"✅  WELL-PREDICTED  ({len(well)} samples, F1 ≥ {SCORE_THRESHOLD})")
print("="*65)

try:
    from tabulate import tabulate
    table = []
    for _, r in well.iterrows():
        table.append([
            r["question"][:50],
            r["ground_truth"][:25],
            r["prediction"][:25],
            f"{r['f1_score']:.2f}",
            "✅" if r["exact_match"] else "~",
        ])
    print(tabulate(table,
                   headers=["Question", "Ground Truth", "Prediction", "F1", "EM"],
                   tablefmt="rounded_outline"))
except ImportError:
    # fallback if tabulate not installed
    for _, r in well.iterrows():
        print(f"  Q:    {r['question']}")
        print(f"  Gold: {r['ground_truth']}")
        print(f"  Pred: {r['prediction']}")
        print(f"  F1:   {r['f1_score']:.3f}  EM: {'✅' if r['exact_match'] else '~'}")
        print()


# ── CELL 12: Error Analysis ───────────────────────────────────
poor = df[~df["correct"]].head(5)
print(f"\n{'='*65}")
print("❌  SAMPLE ERRORS (top 5)")
print("="*65)
for _, r in poor.iterrows():
    print(f"  Q:    {r['question']}")
    print(f"  Gold: {r['ground_truth']}")
    print(f"  Pred: {r['prediction']}")
    print(f"  F1:   {r['f1_score']:.3f}")
    print()


# ── CELL 13: Save CSV ────────────────────────────────────────
out_path = "/kaggle/working/vqa_results.csv" if os.path.exists("/kaggle/working") else "vqa_results.csv"
df.to_csv(out_path, index=False)
print(f"💾  Results saved → {out_path}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'HuggingFaceM4/DocumentVQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'HuggingFaceM4/DocumentVQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Using device: cuda
📂 Loading dataset...
  Trying: HuggingFaceM4/DocumentVQA ...


Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

  ✅ Loaded 5349 samples from HuggingFaceM4/DocumentVQA
  Keys: ['questionId', 'question', 'question_types', 'image', 'docId', 'ucsf_document_id', 'ucsf_document_page_no', 'answers']

Sample 0 question : What is the ‘actual’ value per 1000, during the year 1975?
Sample 0 answer   : 0.28

🔄 Loading Qwen/Qwen2-VL-2B-Instruct ...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

  ✅ Model ready on cuda

🚀 Running inference on 50 samples...

[  1/50] ❌  F1=0.00  Q: What is the ‘actual’ value per 1000, during the year 19
         Gold: 0.28
         Pred: The 'actual' value per 1000 during the year 1975 i

[  2/50] ✅  F1=0.50  Q: What is name of university?
         Gold: university of california
         Pred: The name of the university is "University of Calif

[  3/50] ✅  F1=0.50  Q: What is the name of the company?
         Gold: itc limited
         Pred: The name of the company is ITC Limited.

[  4/50] ❌  F1=0.44  Q: Where is the university located ?
         Gold: san diego
         Pred: The university is located in San Diego, California

[  5/50] ❌  F1=0.33  Q: To whom is the document sent?
         Gold: Paul
         Pred: The document is sent to Paul.

[  6/50] ❌  F1=0.32  Q: What the location address of NSDA?
         Gold: 1128 SIXTEENTH ST., N. W., WASHINGTON, D. C. 20036
         Pred: The location address of the National Soft Drink As

[  7/50] 